In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost', 'pyarrow', 'polars'])
import os, gc
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import xgboost as xgb
import polars as pl
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pickle
import duckdb

In [ ]:
PROCESSED_DATA_DIR = '/kaggle/input/datasets/b22dckh072/file05' 

META_PATH  = f'{PROCESSED_DATA_DIR}/filtered_metadata.parquet'
TRAIN_PATH = f'{PROCESSED_DATA_DIR}/train_interactions.parquet'
CAND_PATH  = f'{PROCESSED_DATA_DIR}/candidates_phase2.parquet'
TEST_PATH  = f'{PROCESSED_DATA_DIR}/test_interactions.parquet'
FEAT_PATH  = f'{PROCESSED_DATA_DIR}/features.parquet'
MODEL_PATH = '/kaggle/input/datasets/b22dckh072/file05/xgboost_ranking_model.json' 
DICT_PATH  = '/kaggle/input/datasets/b22dckh072/file05/category_dict.pkl'

FEATURES = [
    'user_total_actions', 
    'item_total_sales', 
    'price', 
    'average_rating',
    'rating_number',
    'user_avg_rating_given',
    'item_actual_avg_rating',
    'item_verified_ratio',      
    'item_log_helpful_votes',   
    'store_popularity',                    
    'main_category',       
    'sasrec_score',
    'lightgcn_score' 
]

In [ ]:
try:
    libc = ctypes.CDLL("libc.so.6")
    def trim_memory():
        libc.malloc_trim(0)
except:
    def trim_memory():
        pass

print("Đang nạp mô hình XGBoost và bộ từ điển...")
model = xgb.Booster()
model.load_model(MODEL_PATH)
model.set_param({'device': 'cuda'})

with open(DICT_PATH, 'rb') as f:
    cat_dict = pickle.load(f)
#valid_stores = cat_dict['store']
valid_categories = cat_dict['main_category']

print("Đang chuẩn bị đặc trưng Item và User...")
lf_train = pl.scan_parquet(TRAIN_PATH)
lf_meta = pl.scan_parquet(META_PATH)

df_use = (
    lf_train.group_by('mapped_user_id').agg([
        pl.len().alias('user_total_actions').cast(pl.Float32),
        pl.col('rating').mean().alias('user_avg_rating_given').cast(pl.Float32)
    ]).collect()
)

df_ite = (
    lf_train.group_by('mapped_item_id').agg([
        pl.len().alias('item_total_sales').cast(pl.Float32),
        pl.col('rating').mean().alias('item_actual_avg_rating').cast(pl.Float32),
        pl.col('verified_purchase').cast(pl.Float32).sum().alias('item_verified_sales'),
        pl.col('helpful_vote').cast(pl.Float32).sum().alias('item_raw_helpful_votes')
    ]).with_columns([
        (pl.col('item_verified_sales') / pl.col('item_total_sales')).fill_null(0.0).alias('item_verified_ratio'),
        (pl.col('item_raw_helpful_votes') + 1.0).log().alias('item_log_helpful_votes')
    ]).drop(['item_verified_sales', 'item_raw_helpful_votes']).collect()
)
store_counts = lf_meta.group_by('store').agg(pl.len().cast(pl.Float32).alias('store_popularity')).collect()

df_meta_feats = (
    lf_meta.select(['mapped_item_id', 'price', 'average_rating', 'rating_number', 'store', 'categories'])
    .with_columns([
        pl.col('price').cast(pl.Utf8).str.replace_all(r'[^0-9.]', '').cast(pl.Float32, strict=False),
        pl.col('categories').cast(pl.Utf8)
          .str.replace_all(r"\[|\]|'|\"", "")
          .str.split(',')
          .list.get(2)
          .str.strip_chars()
          .fill_null('Unknown')
          .cast(pl.Categorical)
          .alias('main_category')
    ])
    .collect()
    .join(store_counts, on='store', how='left') 
    .drop(['store', 'categories'])              
    .unique(subset=['mapped_item_id'])
)

del lf_train, lf_meta, store_counts
gc.collect()
trim_memory()

print("Bắt đầu quá trình chấm điểm hàng loạt...")
pf = pq.ParquetFile(CAND_PATH)
reader = pf.iter_batches(batch_size=1000000)
writer = None
TMP_SCORE_PATH = '/kaggle/working/tmp_scores.parquet'

for batch in tqdm(reader, desc="Scoring Chunks"):
    chunk = pl.from_arrow(batch)
    
    chunk = chunk.join(df_use, on='mapped_user_id', how='left')
    chunk = chunk.join(df_ite, on='mapped_item_id', how='left')
    chunk = chunk.join(df_meta_feats, on='mapped_item_id', how='left')
    
    chunk = chunk.with_columns([
        ((201.0 - pl.col('sasrec_rank').cast(pl.Float32)).clip(lower_bound=0.0)).fill_null(0.0).alias('sasrec_score'),
        ((201.0 - pl.col('lightgcn_rank').cast(pl.Float32)).clip(lower_bound=0.0)).fill_null(0.0).alias('lightgcn_score'),
        
        pl.col('average_rating').fill_null(3.0),
        pl.col('rating_number').fill_null(0.0),
        pl.col('user_total_actions').fill_null(0.0),
        pl.col('item_total_sales').fill_null(0.0),
        pl.col('user_avg_rating_given').fill_null(3.0),
        pl.col('item_actual_avg_rating').fill_null(3.0),
        pl.col('item_verified_ratio').fill_null(0.0),
        pl.col('item_log_helpful_votes').fill_null(0.0),
        pl.col('store_popularity').fill_null(0.0),
        pl.col('main_category').fill_null('Unknown')
    ])
    
    X_cands_pd = chunk.select(FEATURES).to_pandas()
    
    #X_cands_pd['store'] = pd.Categorical(X_cands_pd['store'].astype(str), categories=valid_stores)
    X_cands_pd['main_category'] = pd.Categorical(X_cands_pd['main_category'].astype(str), categories=valid_categories)
    
    dtest = xgb.DMatrix(X_cands_pd, missing=np.nan, feature_names=FEATURES, enable_categorical=True)
    scores = model.predict(dtest)
    
    chunk_res = pl.DataFrame({
        'mapped_user_id': chunk['mapped_user_id'],
        'mapped_item_id': chunk['mapped_item_id'],
        'score': pl.Series(scores, dtype=pl.Float32)
    })
    
    table = chunk_res.to_arrow()
    if writer is None:
        writer = pq.ParquetWriter(TMP_SCORE_PATH, table.schema)
    writer.write_table(table)
    
    del chunk, X_cands_pd, dtest, scores, chunk_res, table
    gc.collect()
    trim_memory()

if writer is not None:
    writer.close()

print("Đang thực hiện sắp xếp toàn cục để lấy Top 100 bằng DuckDB (Siêu tiết kiệm RAM)...")
FINAL_TOP100_PATH = '/kaggle/working/top100_final_recommendations.parquet'

# Sử dụng DuckDB Window Function để lấy Top 100 thẳng từ ổ cứng không cần tải vào RAM
duckdb.sql(f"""
    COPY (
        SELECT mapped_user_id, mapped_item_id, score
        FROM read_parquet('{TMP_SCORE_PATH}')
        QUALIFY ROW_NUMBER() OVER (PARTITION BY mapped_user_id ORDER BY score DESC, mapped_item_id ASC) <= 100
    ) TO '{FINAL_TOP100_PATH}' (FORMAT PARQUET);
""")

print(f"Hoàn tất! File kết quả đã được lưu tại: {FINAL_TOP100_PATH}")